# Uji original/picked-v2: riwayat eksperimen terpisah

Pengguna melaporkan original pada uji ini kurang cocok dengan harmoni dan picked-v2 glitch. ZIP uji sudah diperiksa: kedua varian memakai piano yang sama pada detik 77.130625–107.130625; parameter aktual hanya berbeda pada caption dan path input. Hasil ini terpisah dari kandidat 90 detik yang kini dinilai berguna serta klip awal 30 detik yang dipilih sebagai gaya suara.

Target aktif adalah strumming dan petikan **bersamaan** dalam gaya contoh yang dipilih: gunakan notebook 21. Notebook 20 tetap untuk reproduksi, default `DRY_RUN=True`. Mengubahnya ke False mereproduksi dua uji 30 detik, dengan runtime CUDA/cache yang sudah siap. Tidak ada render full-song atau pengulangan bass/drum otomatis.


In [ ]:
from pathlib import Path
import base64, gzip, hashlib, importlib.util, json, shutil, sys, tempfile, uuid
from google.colab import files
DRY_RUN = True
if not DRY_RUN:
    assert shutil.which("nvidia-smi"), "Aktifkan GPU dan gunakan VM dengan runtime/bobot ACE-Step yang sudah siap"
# BEGIN EMBEDDED CONTEXT HELPERS
EMBEDDED_HELPERS = {
  "electric_full_song.py": {
    "sha256": "1141f65172b7916649d87ac6baa82cfd9115a9cf7afe853d3c2b3bd8bbde1b16",
    "gzip_base64": "H4sIAAAAAAAC/70863Lbxnr/9RRbnB8BFQqk7DpnyoSZUWQpVo8taSw77VTVYEBgSW4EAjgASJlWNdN36Bv2SfpddoHFhZTipvVkYmLv+91va8dxblSyiOVRFhSFOB7/93/+1w/jcSHOYhmWuQrFL9A+FCoJ43UEI8W6kCIoRCDiNJFincVpEMlInKZxMBNLGWcy9w4OPheyEOVSCvlFFSXOy1SSwLh8nZRqBUskkZDJQsEaJ9cXQ/GgymW6LoUqCxGpYJGkMC0Up+8vvisOjsej13CmMMg8cQUzinSdh1KEaVLKL+VQ4ElWaSRjGBLHP4okhSGwZ7hcJ/e4eZoLWK4Ml/DhHTiOc6BWWZqXIsgXWZAX0nyHabY1v3+HNczvVVAuze+0OJjn6Upk0BarmdDN19aQYj3L8jSURVG1bKufeH9eodxmACbdfAN/x/IyWMkiC8LqROu1iszvh2BTtX9V2VzF8uDg48mnMzEVr1+Nx+OD649nN2ef4PPxQMAfpwiXchX4G5kXKk2ciTgeCkdF8MORGsNHM8CwA80EQeyB3YtSZkeb4zfYKZ0hLxbJjQoljgjXUYBTslyGSq/szIEUytevsF0lRZmvVzIpscNsUEgZFdBwO77TKy7gckESSr8AxOHKf/XGNH0uc0ntcA6c8gab0zluEfll6ofZGlo/5WupVwLaKPU5LgqgxRJobQ7YhouX2xjo0NAzHkbAvmWQiyAM01UWJApP6okTEeXboUiCcp0HscjidXgPy5RIXg5tY/1xkGJFGMsgj7dAR0Cta942KMsgvAemKZaIqCQtpchh/wB5Auk+AE6axcg8wCxiGeSrNFFh4fVsciIKIguxWhcKYETHj5FrVAI8mKT5Sjce6TvlcgEcB+vO0zhOH5D6kQ8zFSTpd0XPFrw/3BvoEkfjCaNtEqzwSID6p4ODg0jOgSuLMAVC8vM0LV2ki6EA/AOmchlNLwFIgwnjwnHOFV5TzHIVLYBPlzK8R+Y2TI4gmaXpvVjE6SyIC+RPoKQ1EOlGAjSBGpIEz2KYGlaAszh8eNwaKBz5jY4x8HIJKN9Id0D9qyC/B7TDENcBETFXi2IEAJJlQdTsIV8jOZZpGhcjgNXKl5sgHoFo8rItEWqYq6yEPlmuM1+LLOzjDRAaMEFFLooAfWv8k8P4HPASxy51iREQ0woOqAofmdUdAFZyakP86YPyompeQ1OoAmEkEKj16rSivnc1dODJL0DBEYjl3G1DQq+LS/Wel84cKADnR77jWZ6nuTt3Pl5dfRLbADAQqUzFailm6/sgqTHJmJ2IR1zxyRm0QYDN1AaYjIDHS4n4AHjqkyEYgEAjoFMARKTC0kOJeC+3hatpi3AFwhlYc2CdGWcSJGCey0sMzVIjQB2pBsduugEMF7/KROYBaqKjD0hUhRZqnT+HPNFD0nQdVCOrIPNR4PmHI736YNgaBhLT19rI5zH+oTNoQVo1qcYCjRdE3Frjj8GUyyDaAuBu6cb21evJzWvAJob0nI1MojQfnZyeHd2gPD/23ow8aNyMZioZZVtgxsSxiLO5EsqBvUsRMWSpSoBT2ipjRJzrFcFcljIp0rywN7pj0limoE6ILkBOysjl28Il68tVvBHLxNUTBuJncWwxXZeAnZMIhI+cAcazoE20nrgolEAK999dXH4Cbk4WQNpFEAdL+H+5NjBWk660tP98L1BWeL8DBFygERdU3tBcalAhkNhBt4Liw7uYiwMVSuJxLWEJZj7jxUUpqwmIWwBO2LYXsQ70us6NFl88DwSEdHDbtPBI8kynwgHFzNs7NikcWAJDT65wth/g+ksL7JmM1ysRABIy/N9vHxCanvhnAHGCYqSAMSRbK3MQrAAxS2ew716QO5GqlEcxOh4D04HZ6Rvl7s/Xcewjz3oq2yazoYiDeC1AL4OlKRYkAgrlOQ3c8EU1Ckh3L23gg+HmIUHcTgB5wImAZdfgoaVBUC7Y3VqNOEDvvPo8R+vOlsJkRKBZ56UZUDjOpl4grHzmDMjSBmshndjy3KUmbyFLWhDEGuBnKKrWJFyC/gT0NlqLYJU9qAiUx2Ag/gGUIxqOQ/EK/utVCb8B6Ax6z75kAGFQTGhZyBSMzft3X8X16YfjH7qyvz6Hvu9AXx/YGg0dkIFfpQ/uw8JlWQmmB9jhskTogSE1BVsvWqOwTpPpMXwsApVMx95fh6Axv/hVF/gptcUBJ4xVqEp0SxZHWhaD98E2dAFHTxOwLGZbsIZAuz6kqHp/BLGyCMKtiMH0AccDtt8CqQIJlJW1QYY6EAEimfBt7PVc/n2tckmNNHKDACOzg65RX4Jv0Dx9xWtukGxdsKHBYkZL2N2A5oMVBygHkQ+bXS6cDnQbWtqDagg6J8CoYO4qoIQNGxkb1BJ8pKZgJ18Izid+EmOzwhg+zMnET9PGSfEbQG0Pxetg83FHKthEAz6g4DPxhj/biP1pau/xEyKTNA4tjap9PDy+07RVe1MeyAv31oa868znq0wukPecI/QawUtFuXy0wf9LPAp9cqNymEWZ8Ab9NoBzRP4KjcOD09KlbqkQiI3BHC0NZ5OCvJPTRzz8k7Nr0SDXSyDj8fQQD0Ue01E4YY8qXPnF8Q+x1GOZMwZ3Q1ZjU3R5GCpApGuw/adGquiRIHhwfUNewaxw9cijCvikQfXAPRicO29lmKJ3X9HCShVAbeFyQuSPOj8Sj6b3CQgdKOSR93tqillu1JIAzFe07Cr+B+2crXv53/Ir2OnOwQdJkB0MHxbL4NWbH4biIQdK88lp/2N8C1CymEzz7i4e1N0v4EMmnQa7PcctQZtfapVcYcByDdhEsw5XC5xdx69HNG6wT+dWUqN7xZqedPfxGMVCS3I8KyQayCbhhR7rwzKNUYDESB95DRsTowILAsR8VGgYpblaqCSI/WVQoJvEZGEYncGYzFPoQRLxMJBQuJZkIfbymRC7IgZGzSTLmJZcASf/wQcfJVdyp0/hzNFPL6fmojQxRdnhaGfUkknA6qi7mNM1q28CBXZOjG4vYczFq9zqZZ27W6da+W4AfE7kYxtzTbRVyxHe6sV/QvxBi2sTW0Vo9bifEave+M1exN6QUdbBlkjnbCD8SEhGg41wDEcE7Qw7wWFBiONJVipZgwvwAh6oj/a9wINZ2mzvETmMCKtRpEYi3YEdiueqpZv8Eso8Kw2VEX14q/tIoaWGzmlBmBpypNNP76fnARjWPBzDQ2i48zS0CQ2ZemD0sYERpqsVXgJsyz9RsXWV2N0+ME6qY3wP5+jRdneNs/Ig0n5G+YF9Rgf5Ji1HcNJ7tLS93rKr/8iXrtUfL2ETPZDQIak4lEM8Wpsyh89pvkrxEali8K1E4twhevaSJuhf3pvVbUsLI7W+5ByaCju0WtGpPgq7kIbXwBTlvb83ex+DfWAZCOCeUVB/2pp4aMbTdEYLhzV0EKomaG6vybnHo2HUWC6NYYJha6C1Bw5/0MPxlDKyA2H47YEHCRwYrArXfUV+DDs0GCm+vLo8QzpDVGCINwdqQhjViu5hiUJnFqfhvZhMqxN5GIPQFEUEPDG7kXnBPXnw4NLMRpRNw3LnBOffxw4A1jUwPxT/qM+DMFdIEYyKRx21/60RtNeYhq/HWtpPmnjD0CXyOrTT3087TNGZDEoKxIOucaL0IWk0hEBgkf7qn/8Q5HhWGuOcIIsckXdlX4SyMCoBYw2lKMeWR/dyO6LlxUqWQRSUgefcPTHdVOabWxOXvaCJ29ptg91TtYkZmWmP/bmQnaCsbl4foiDgn2COwaL6p6E1xGezoxrYsEKYsJ/6loa71FP1pH1gGDxpftdgJLKpTKBqpYZRhOdETuxspPmzg2t9y874FptiYCGWPoYhYAwzoUOCw2fyh1b67NnBAr8tJU02yt9D6TKJfM1L9T66oe8uNmPU/CQje4/aPDZ3R5cOOkC77SFTlaBRoAnNoKTh/ZhG7f/ILzIE28bnMAQFjGon6JDyKiDdyB7G3KtJpk1NKo2i/5QgIwWlB3I+tZWK+SgxY4s2Dayjozg68Qo2DHzHW86+Cq11j0CyBZh0oYTa6Pz69auRjqNg+idXkZWL4YNiVoITj5zFwCZjOlKUkdsqb6u6ELsn4O1XOcIIOugHp7McMlF5OoaUXDu1SGGs+nufHv0XArS9M+l08mVbBh9vpvWpHQ/c54Sy97DOYHkZrPqdUR3rBGdho/I0oVMYn7TbRfPsodO+UfbJrMyS67w79999/sU/PTl9R5oQvv/17FP9/eH6/S8np387u3yrv06vLs8vfn178dGxzPq08PRmt7g0BkCtzbntoDnQW2cYuXf1Aa7Oz99fXJ5NnWPY59PHk8ub86uPH84+3tg9Wg02YOjuCXbjkQPMwzQitB5gV+YlGgN2FLZvvla8JiGf5uHSthypwcOcD8a/K+/iuSD46ee3J2Kd1N4I5vgoIK4xRzrx9Prz6MP1jZgHcTwLwnuncRadSPHAuItioFgMAAOb6097JM6FqxXm4JqpNd9ZAeOKxHAxHnWgvRPMhZ9z2v4d7+Bau3knoUSQ6S7r9ijDfMrFr0A++iyakcj9OinkFjKegzQL8gWI68PD+wf81c2JcTszt64wIMbmIoNJL6TtmhUtb7TvVAjEgtOMaOBJvAirLYCAGbm6WKExLMvx2M7ttalquBP2Rm/VpxF65qPfTs6mdbHDPF4XS8svaUXDi3VGmdlnwdUGVBPYAB5/E0ifbqFBy9DS8t7s173ebowCNJrYb0ZSNIXBKENrOP8X/m1RkKkV0j1ue4E24fgYhvD9FxFIBT4zaSeYGkdnx4BSzoB6/vIRdI3hFIpkF2L/KfAPd9xWlSYoDJn2rLbOLCDwlrLuT+737dSaSDuSZGu2d6N3mhRa0NgPujag8Hb0pWsuOgQAA1qIp5F/Eacg2NCT0UaOCz7cJlXgVZpiq+/sEi/2HKjC64iNO87GeFoBorjy7MwR7NtJJBlbhCz+hntqGtlcWegqgLQZlKmb28N0pKcvtgM2kcyVVs36nFkcJK7ZUisp24AbYu1Te7rRmJgQ+T2dGRMXUyKGx/3KPJma2ivkahIkU0OEXL+1w2cD3mhUT+mgVbMAa4opNlhl6hQk93yUe07HybKgCMCzLqIt4LpFG1FAbACjVoGbazT9dL+m16Q31cCsJSZbvdVtdwNEcPUPCN9EzQGK1YZ7qoO0bNXQPmgBUUMPKGhaA2NYUeC0JoE+pFartaGPxW8oStY5CHqK/ORTywvkHgQJFTJQIda0NgDLfFvLE02QhgeZ89uIwahSVmKJpzyjn4hSNBJQy1prdUgVnKhyXUydOVg6wFtDnjGdO4+knugLZTYaiL7/BN4wNTUKhVCts+GKMiq2zv6NpGZcKzR3WDJ28sthukYmpITysZ1WelGi6P8jD1nlFAMQrKspOrnsX08f6fBPw1aycX/I1Vr3DycY1bydWgSrjA6xz9O6YnqNwJenMhLkxXwjdTFwEJaVWuLVeUXHoA8kw32wkNq5/sMoQieoiXRyh9yq2tWPsToyocjNsC6C9eGcGyUfvFX2GlHoOiv1hcpJzEj8bgyyTIT/A8Ko2H5vprqB2VjN4GAxwJRWmnHr8T+9um+vuRvnILcAriit+RcV/xRu08PURSUaslXNDSipYh2X3leVOQjBCrbo23cA3sRDA9qDPbe195tLGaF26O5oI6oXdxbyKPahq6m9f1PZOZY7VaDi+zpfKPDIgWTMVFbjL679t2fn74Hx3nK1DljfatMy7OZU3xqDHNuwe36o4YzWL5lwEzORw8b1/mbasPplOKWJoIkdbXJt/V3VuxqlNGqEqyzjZ9SRr41b2CMLhkPfIuPx2D8hnPtY8D0eabIwQ7UaG+m/dTzTVEiRQqkw2xQDjuP8TcpMBKJYgcYQGLlGbUy+NoNDBHPMjVxdfRjhUrB+/bRgFRT0GAArfcJgDbq+imK1kxo7aKxRH4cRB7ILC/c5asKxfwoRmfhOH+7bVaANEraKL3eSGpM6jdbY0FiVOkDZrtY43FWwMTS2v7aXNXH2LIKZpyBzQWjNsFpwoqubrUJm4Ou+iYM/Fp87vfrw4eLTzXBvtcg3BOhYM5EZ62sVYE0B32SxkFFjhypiiEXtEdqGNlXpvIKFLNawSEaXaXkOujIyxTk6fY3T5tgOVhZPN2YW28G0klU8aozexvsP5kKrEN1MtErXMZ1p1U7YgzBr5mNsRNcWcjR4n5nQeF9kYjiRms+xcp8fx4DBUNAhwaUzSZtaEWkdG3G488+u85+06+dNONFsO+jiyL4eFdCbtHHjHZR+E6GfMHEIPs2331ZfoIUe5QzJJMfcFMt52Ms86qGEleXQsS7AIPeEHjx5EUhV/OEy3jrq1zGumo/ZeIXzNEPdkleGWcOWj9VJGj3rc3V35TgZPukqagdY5+2wChjvhc/BsNpTJet0TTG6+lkYJSAjmYERjaxcPwlDajCvwcRvJ2dUfFIqitiC+iB3xBPvgjx6ABSAg1IS8Zs3OLBBQdZLuMVnZaCJEom1AlxmV0pCtOfsSRj1qGdGZY8nZ6X4+svmGmGOoWgVn9rlubhD24XDkEVCE8ksmDbTV996A4tztCqY9IRa5871+5PLCb3fezQb337XTP19d/dUGJxSAP3X68+ieiH2fACWj3XQ3vnq8kxcn9zcTOyd6z0n3uv5ky6+Hj5zOB3jZwqtXx9WNLbjjFbhD1rn3ep/XU5OKtEnYeP7lmYkh+KIU4hk8R/hLKfOfvAAwFnD/h/UQUoiFQqYseCT+dwndwxDrjXdVIqsLsYBPTjdmY7quhU1+d06fGATEKGgZs/upopNa89OwM1SRG5fIK9jyw4aSsomUmskJSG0JOUkhDGBIgcTkfgKpR5+64D2LrDkDkYev7iGPiWRhT49VdRbGQYmJJVs0pCjkfU5YScMM3b2vh3fWU+j6MlTf2SzY5c7FmbQQKe4qrFJqpdV8HtIoeoGyPQMk+MldUxG/o4sDo6AlW5bFQV3OKJirXbncwC1czP6QLwzOzhoM6i5Ykhqx8ICKAe3qZwE9BE5vLgEnEKWJRW33N06PAj1W18vT7WC/tKgGBVy/SKVtaqVJdBvYbGuYNj3CHbXE9j+ahy7RqKCZat8Arcswf+h0D7qzVgu0uZL3SBuvKjtbAMGNfhZPj85VV/NlmSN4Oog6dC9sprAxQKGLv0aEtzTv74ZjXcoelbBRzzrYCFN15NNkPiIgpFFJHgvt0R99P6BzEVoGepPMBgNojyFbwPdwbMPYE5Ytzcel9B+wMN5oQ3XHruV7ayRKd5qMBGTj5Y59DSazoyPxPANnNU7wzIFv1BfNVMdtwcgjHIQzOmKeBtG6SQTQeq5y31OKsK14q240Ih2FhXRW8qLvfedkrjfy++XwWThoRrEH4Qt+kGvHHk2XZKGAc3egsBF8MGnAQ9lCODW+LQNNCimQDFbarEmesSN1RpSXvewkG8M03zUkWjtJzV9k4jXeE4jK+uYcnm6UXslJOVQC9K9K4Q9F2GbBy9CYGQYwiaDlz/xqlUTlv6wThrpbIS5ZpOQ3WatfVOot4vS7jqPkdrVblWcyiqyG+zVFC9fslF+3Vq0URx39yzALkjJ4FO7Bb+QQe/O1uZWVnFgq4g6PVG9Ua60CzqpaOihwK6P3xdJrc/dKLW727HV/sV6ehvx2V2b1W/znrshxmA7m7Vi7M9fqd+FqW20YSWqpl2FjUk9omJfU/FU/70r2mzpYJ29s62cTq+zM2ydyeCen5ADL6JtB0bWbAuOYmPBnaNesrBOs+xftzlo97JEhswI0w4vdekHDH7E5K4Jzdj+rj2Z3rCY03ruCTPLHKS7aXszNOTFtDF9nmS+1XttJaIOdnuVL8yi9tPu/yKDqgPcu+z9tgNkS2w9tRHDarpn/EnemaXgzbxeNf6tkMZox67AvyVVCMgVgLHUF2Ss+Iv4CCSg+NVOlVmseHPEkp8tGmkme38wC/2i6+g3/HAWE5GnYhf0x0zhi3eSL8jWvqYeN5IcgcS4ue9HaQj+vTUT/1kIP9BT3Nq/RwqZXlNOAbYMgNGnu2IE+h/QKG6P7/auXP2zGdXae4dzTME6hwmLWj7/rp2oDltPJnOmvsV470zr9Zw9Gf/NqalztVJl6+mgiZKtgi9qtV7hyx9jwe7YQYeqqLqGkAIKOcU8Llxq/8wqCtMzUR+xIoKbz9foEd40imP0uvQXrmxyrujbYHkkb2DXZDQqyGkMkoedOKFGQFX7XxWRDZ/A7FBUr+H4QZg+kWRrx5AI9htkO23FDPeoUkbViYb28vWhTO6IO5sxS26si/FNNoma9cegFdFzbkiiTiqOvDWWMfq+/yG4LgG7zQmQZRWWI7KMJefB5wpNH31p4uSD/wF4HN39Ik0AAA=="
  },
  "guitar_probe.py": {
    "sha256": "af580070d1f10f3c21a3624cee0615968aba54ad2037b0806e78c363a74d4d0f",
    "gzip_base64": "H4sIAAAAAAAC/61aX3PbxhF/16e4uA8AHQii7EkzVooH1ZYznqaOJrKbaTUczBE4kghBAD0AlGiPZ/od+g37Sbq7t4d/BCTHsR5s8u52b2//3W/3+OTJk58zJaJUyexUpSqqdBKdruukklpEeVap+0q45/P//ee/L+blTOSZkKJIskzF4vLl1elNpQrx8v2rS6HrrEp2yn/y5MlJsityXQmp14XUpbLfN7LcpMnSfjX/wYBfV0lqR38r88x+3slqc7LS+U4U8AlWMo24xgn+XG665GW9LHQeqbK0Ix+SYpWk6sR+t6cMV3WahrDbWshSlEm2xkXXv1zdXL0Tgfh4IuDPSWLnQjij+jndnzuecHZ5rFJcJGFXUAcMf3e6lKVyPMMiVvskUsSmjiWSFFpFSZnkGQ6u0lxWz5/heJKVla53KqtwwuxiuZRKxSUM384XHs3FMotUWEYyRd7f+3NisFJa0ThIgsu/mzN9vsKN4rDKw6ioYeadrhXPRbKoWJo3ZZ7KCqxLR26UJdglZBTlu0JmCQrpi0txJ/XOo8XaE3UWJ2UFSgb6IWWRygMMZ7KqtUzTg7hLqo1YA5tUgUdFW9xzk+tY7PMkAmuUQmaxKMmDhN4cqs0OuJGCSl+8ztM0vxPVBolllotC6VWud6gUw6f0BDgkMPJEsdESDUwc40MmgRNMg9B7HIU9gAioO+x2Ks3jgy9u6qUVUMiqktHWs2cQWV4pEatIHoixuge7lmWyV+D5VRLVoEjQqi+uBqrIs/QAXHIBXlLyoCdiPJjH24Ms+xxMW/pg/0/glm9e/u3qVTjwzqdPzYD3sJ8a7Z7unzl/1N6/krVjfXjY1mRatlwC6/ZJXJPNi7TuGLrKM2WszG4gdaHW6yQHNcAU2pqMBplEUG4Rb1nzxhKGlqygQQQIOSDsWKGQNQ2xH03a+MiXPst/fHFtvBYFKFuj9l3VMKi02uVpDqozOgPlNwbvOcGo9W/e/fL+7+FPl/+8+uULPYBEOcUIhKz1/He4gTnMqDeAZUllkxkAzSvyFS3SihNTa0O2uXGFRlfGdzpGtOoeWBHUBHw1S/aoDYnamtEXP46dBy1oFI7R9of0jRH3Jer+mnq+MfHXJtxekjVO24k9CgYQAe7KtDyygg0lmzJ1Dpcy2aHvIl/bEKTsG1S/k+tknWQSr9rGBuZIONLNj0bf/Od0fB8WHscSs2lWHBkf0u9JrFaCrk+VgRDKhfNXswvaxyAHn1LSxkzQ+J/s4VDVyzyHdBVJrQ+kmqjWGo9q2JEm8zQGf46ACRipzGsNnr7USbxWpc8MfwIJRGIUZNwiKQXDnR8sAmNaT9ypZL2pjJGBOc4DzqkkkoKsleGKyAr0i4jKDUNESmE489H+IQSechmGhEDtFwfHnC1Z4amI1k9KonJZHUOepBBxJpwqz9PyDHDJLlR7mZ4xQ6PDQkVA0EeEPo6GCP+MWCmkRAygRiQ4+VKxRQA+4a5GPNbqEUOAajUwIpbI3MV/Zo0EPlpYaV/dw75mrWtYmTVaQShlzJ2dAhLSVq6Vm9dVUVce4M8IrcHK0Ar3L0GS257QPqJcRoIQpio+SzIgb4YhJpWmw55BECptrl8z23VucvDO4hLDFSDHCJ/5fB5ehiwEgsn5GQR3nU6yxXRVa3XG/5tlC1qG7orOAVdrHSc5uuJtY5UV8F6CVnwA3iBD5+CAg9WdvyueE2xO7ntDi9mRCM1f64RG4j5n/07umSN9XMwWrSuip4KkIgjECBMLIKzZLnoiYP2TZLVqBikrcj3h/yspXqPfG8uDhxuNOHcgC2YzbWwRNOvfXIevrl7/dPnu6tUMSw6pow3k1v6eqFoAM2CzPWKnxoO+Naq+ONIRHLAVwVLOxqKy+8d7+3c6qdQxvddyOhls9oA2j7fC01AygJOYTfx1mi9dp72nnjoTIiarYX4ZSI2zJup9FGrGEck+b3K012RDG6BPPcyBugqgXoprExzB8zkCRxXVlQpeA+6C9Xup4fqqgvbe8drbNSRjBG/h5mXpOSPujMSrJEMJLf8ZIjqcPp+LvwTNtvj5xbw9vZYJXNj/kGmtrrTOtev82CvES7Gry0osleCCXJQKpmLI/XBwCWYQlF0E3ArP522q5qOQBGAIvlQf2vd9ts3yu8ziEWbAHNGzVYX53TC65WmTG8a0jvmPLoNi5gNxnu7BnMY5UKBRS80W/VR+dPuaTETNAZ3vVUaVX9MTkM+++zNcgegoIbUUmtV8TQKvfaLzzEAsQzYy1dIBjlByF/J12yGBi2m9VnG7krCMViEi+5DQmV2s/l0nMLEqnj874QyzStblAN+Mg8r2zmDQ82C1N5XVe2jo8Wqht+kjZO0SQ/Wpc0Q4IRoNcQCf2YHP/LHvQOCwZryNfErTyJNu6NLlebBHHGJguLOZ+CZgx/yMeGIPjpMVxDNED2BbALKZuuOZsz0MI1Q3TsaZa7eNE+2iZbOqDKh9AkkDYHiYb03WmHUufbIpZJqqxjYMX/QAFaf7PLQIdocx82FoPWo1IYWBnGGB/kUR3/ShQqga0zJER8NmFELj8yM2JopCEySwwnxwKULNnN8gwaNb2UE4i0L8zP5qYT4kogSzGoJ8QvDirelwnGGhC1ppwUiD+yHks9jjkmRFlYOtEbhcyABNiBSUjMTC4IUfsMxOrjcQWWcvc60u30BZJJOdzy7XRn17s43CL2MpY7VKH1rP2alKxrKSYESG9wzU3EFiO2shnGMvFvq3c7k0t0AXlAyuEkzZmJ7xRulfh8OFFlb3hzt5tUcdJ1A+oC+yjYdkfeyh8w8qg8XN4ToXtQMQZEDtl/VqBaALrKY0xGCPmWnL+lFeHCiIB7Qe73aEL1hQnsWwNme4mA7nhjU4kswgFaPu0Xlwc2f2iDrNToNF6Be3zmCxs6CgRsSBUVNpKyVEXxNMRtpPJ31efl3EiElMQgicIpXYQwefIeAfWIdrhf0iJ+4gEYYzfWfqsmqi0Zlxautrqi084NScFlBsd9T1sfvteJ1kdjtfTPGzytjJ+/C3fBkyigleICSjZnmbwAJumk/WBoxngYYSYGBS5y235RcQqb3Gd9C2vaf/+t314HtPILdgJO/2jzhuswdquY5Shm5Kpaax7UlbkOx2mBsBTqH3cWoyib84VJAQDTSaeeSdg5q+yRDor6end7neKj2uWZhGRo7hQzyJBk7FYwzSaNTCwwY0L74o09nTfYvVMghg6U5N8LEsgwTWbtUiMZc5gXqzfTAC6uyJoo2KtuQRrfZ7bt/BHJ9n0VkPlHTV0Fm5VpVroQFlOMsRIgkrhVRlbrv81oEYKaGqxZXng8vhKCVeAV2EXUGEnQhEU4XfGMgDp47LwjcM7aOdIHRP2gqO2lKdO+HhpoLTSXxYGLa5w3S2MOBdw9RDATxyhMHVSITgJ2+ps6l5xOiNX9OM3ig34ArgZKY70IpVaxISL7p1zAVuwc8C19jkezT5mLL5/jEimRYON1UEIMhklZiGlVhJCMG4o3nImhJ7/y04JioQQFUVvn2BERyzCFPY2KwhdRY9xyXL4x3V9pptPrQj+HqIloKp+ehD4nhyHH9aFI4FN/C9VaPJSM3UYoJlJcttWB0KgpOpWud9bEyt3ul0zSsj21P/kWOI+qvs75WW0RZBKAUEzZgkxBX1xcTFcnTzX4wmn0fSm1BQE9CniV2gaF1Cbs7w3TJNPlhFmg7EhM42SbbFMsIuEw5UL2GUV2Fr84cY2NVorXKEC9zw61qulZ361A1NmR1c45YUbVsKsT1V8lvI/ljMWzf04SrclZAEHwuiy6iq8TmIKwJ2cS7OnF5m4LrP5E96DycBIF9heHdnl7KKNmGZfOBMcT5cgMfVcEdgJxijYWYNR4d+TOb3WRNtjdwqPqNdmyN0ZOce6uSNMtprnbhLuk0Elwl6WmIEbXLZrJ/gbJOhzX+gFyY4xnVnhgd1VWd/KFO+wdw4hOesui4S/UzgYB4Dx0salHMSwN86vzO3T9YWo1IzIgPngO3NnnzZUXOVszg+gSAoI3Q6wPbczG4VvYfMEIcriDmFIODztmp64d0N2rHHuWNElzoCPF5CKbuWENjdJr9tuY+29v3vAVq5ne2Oev2eOJ8NDND+bscHwObeOqvVrlDULTnN8rKKE2pDnRI/hSYyU/Rv0gOlcFYQffYwyHdO5QqoVs4+T+G2CT7iGT8Rt+iCfqSTJkuQNQXN0OjSjJ6/eLYd7gY6mi3G0eR49dfCPa/JFsHxJY/VUARXqQ45zAP+f+po5FnGwYOjoO5YDuyDJplaaS03qcIOFuDSrYvQjmanGRVKbkMEcaFM6XkPAOLyUKk+w8lVzuwr1MvDF7yhAYeFmLqPVFGJvwKouKKPiPBkKcgpLx6xPCNBz6wOVs5HxD8ufZv5IT25huGnC/GRhj45X+GAvc7W2Ik7vcvOAZvDDYnLst9hNY1vvFXSzka/W9yTE8j5VgP0whSGOwjKMOSHJfo9A5Yi9neF/qVeE1K8phk3VmWkE5I5CMM4j6Dq7VD6Mo5DySRuW+KiBYJrekziZ5RgqnL2uft7e754kLO5Hbu8H1xuyuqOHHzHx51sMrUTNhstMf2UsD3F/EHKBp6PEz9/mJrbS0AsCYEHDv4cRYWAyG1XZIKy7RZEmxzqpjKoaqhSXX5NmrUitE8iD5vxqFHweVq3DZCxE3hio9IiaDzt5v01yndjGMIwvtkwX/oPOZdu8+SG33yzQTclWCwREIoZ5IqQfzZBtOASnaey0dDpY6WGw9TTL+LxPgDtEnWhpsm6Q+x01KlFhsdPt91DUhjhslvTNF1Mtam7WhuBfW0fwPL+Jhhd3FEZv7i2D80JPouYR+2wyt1jLRMN/0bZH4JLoyC6dVBDG3VvGr0u4U1zxnFoOQ0xfyX/sDC4PRzq6gwFgVu63GFN0QfGU2rqoeOWHSvwMcDL0vToYH/63SM9a3bqauM5Y8CXskJoPN/oGLNsq2RvxL09blIE9kGZlthe4mPN2rZhELTdZ9NzCrqPxiPCzI6f9Jvv/EuiXjFodc+/zhpVLCcFZZRqrwJca5N6tw8EpUTlUkjG9a4o3ebHC420XnfLVnv2sclM0ovTI3qi5yD7JkVk9lv72wca5i/tTyC65vicXYY6HfPXGb58xGi0ZwDi/g8fRLQKhzAAAA=="
  },
  "guitar_run.py": {
    "sha256": "156fdeb26589aef3eda9530e2d2296f5f2c135c9edf03ae09da7ce9401acc41d",
    "gzip_base64": "H4sIAAAAAAAC/7Uba3PctvG7fgXCfhDPuaMl24kzctgZRZYcN7Gtsey2M5obDkTi7hjzFYLUI5r7791dACT4OEp2W33wkSCwu1jse2HHcc4TnrG8ZOJWhHUlWCmKMo/qML5KBJNxtk7EQlYihQmFKONUZJV8xQpYJVkmrkXJ4rRI7licrUQpslB4juPswVheVoyX64KXUph39ZPEV14qKh7xipsvf8g8M88przbmOZd7qzJPWQFjsE5DYOfWFKClWuVlat7lnWwe6yvYTShkM1LBDsxzXcfRngLP6yjODfCQZ3kWhzyJ/xJzlsSw/QwYEaTxrZ4dhiIRJa/y0qxJRZqXd4GseCXnwEQpqqA7JmFJWAWRuI5DACvvsnBTAp6/hAIKZAObgyrPE2mgluLPOi5FQOSpT/NmEF81c8r8WmQcmG8Wnnx49+7tp4s5Czci/BLUhaxKwVNAu+HPfvhxzm7KuBIBcX1vLxIrOlG3gNPnpYjm7IqHX0QGD3EGS2s8dtyCiOTsaI/BX8qzeCVkxXw6Oy/JeSTdBgJ7yhzz7OEEZ+YBBVFQidvKnc0IRrxqwHhrUbmOBHJTHoBUyRiXsO98dqjw4V/JYynYP3lSi9OyzEvX+ZzJusAdA0aFjlewkilAToNGbbtDXZhnUYyT4Wg1hYjOEHTZmRAoAM5yipgTawEwnmdrwMRXFeiIRVtLVMtaluWgHBlznSsupTNnTgQf6GFdxxUvndkU4ouGBy1ICconGIKbMwKGSm6AjVDg+w0uxrPICACyxOGhcKbwv6F1LKxLsAAVGAMtopJVG8GOT04XF5UoDMgWPe6aZAppS0Tmgta4SsjoLNQQvuIEnt25uOKA/ezTMvYze/bkyfNDBtqvBoCFtozukBl4r7P4z1po5FeiuhEiA8C48WeHL16++On5jy9eakIbOZAg65fOsbNE2g1/fM0fJhKAi8bCHREh6SwVsD/yKwKzpDckvJmD1LcL2h3IvC5Bs31mSe84isvmZdmsNmwmIF4sg1WcCJcYqsfgnPLkGsY8hA6i8F2Lqv3Y0vOA8GtbmtZgG64E4wwR4t5QFpothElcsAhkJAQjeqc53VVWRd4upWw00tr019L4kI6aI+rIVhcJnqfHiwJEwb1vqXOO2qMEJW71DD7YFtVBkDCGP/MO4M6fo3iBE6vS7Z/bbG4mGKYc9Xg4BTqq9Z6PLDY3g8utYkUpqrrM2H3fRB+xQ8QOLq6W8OKgG8lgSzBo1P2ocSYdKhzjlIIwT9MYOaO91qWevwQgyF/4gj+95WPm+eiRhr4PKhEcBGqchTYozfkbfj2EYZMDLxJOGPfuKFHLs+TuFYjkDUYgMOVpuMlLsD3AVx4Dw0jOxDUIK1cyg0qb5pFIQF6KupLOVjvqkMMJRAFwxG30Z44MAv8crzM4iFJobSXN81s1wy2A1NRJpfhg22Gc64lbsF/SVnZ97u/zTGg5oPii4/Rp6ZhzV5O1a1ciQursrEWG4RMyaCcmZEjGU6H8Yvdwuipl1GhuCfPcOhHbeXapQvhEEbCvHXjYjEDEK9Ddhps6+wJWLsmzNTi7HFi9ojC40qf2itUSbWAmblhZW74ffW2EEVsgIdAE9wG7BCJsdoQgIIDD77Cxt2ze5bGoKpAtjBvutzM1hiYt7YwMQLz0DmazLoc0bsWZy/6S5f+EQwboTh7ZW1PpCdl9I+RKlprX6cjsEfSAvKzitRagcaIoEfKZa2tUV8ghgkd1IJmEF2UqZq2t7oQ+OAE8cikgg4mvMaJvIVvmnXy1tuewhLatsF4qjI8KTslsqB3EkqWxxOQOQWsvCNZJrGoaBLZI2IQO5ZARYJ47fkCh1xZJp44uZHsQbLZ5orFCmKJok95kNseh+MWkGBcVX7dvNyJeb6ogziClUbYNrCJICqQrRmIIrHmB8xhMcOE0eAWuUpE0JfIjKZY+pasyjtYiKPMc7R1mnH2w1gyASTMCOv4gmA3CKnn5bKlxQqrnYULrgRETZeUezMmv2wjR2YRlXIDh14uIjSCKmMEGIruOIXmkyN3kisNPtM6e6o/N0tvt2ttfz4JfP/8SnByf/HqKwgzv/z791L6/O//9l+OT307fv9ZvJx/en7198/rtR9va5tLTiC4R9BIIsBCrMRVjd9JU4q/XJq30qiWo0SB7kKJwiWKkkhcFgIAWeYw8kuRYKI3ofvJoz6BRKwVgcU8TWvey9SRfge3IZF7KB/KwT8dvTrvJT8qrcEOKtlrFYcwTO+eyCESpQUq0mv2NfdoINECQwyU5CK1dfvHYv0hHGA8Bk9TJg8QpCeCoiAVSBbNCyY12uDDR2+sdjDnqD2dnv799f+rgITmHzmDap4/H7y/OPnx8d/rxYnSyqbDkZbihAVXqYH639OHSBH2oauhRZ6qhYfgQ1hEfJKQflWTrowD2IacLwAtWTx0Mj3iBgX5zQCefXx97DPPBJkcFTh4X4CvYRQy8BHYZd2TMEWynb6GmxHXelzc9ACljQOHd7r13s+5vshmN3SBUAZcQITQmGEQTJEpXuUyZzKzS72pral1g0gO9BbPhFpUxLqRSsAPz7rDvqdjm4T8vwCJuxO3l0U+UR/dMKoanKVroMw759Exl1QaQ08fEfHsXjyFWv+Y1ssumdzZmtByQi4tPp+cBWL2T384/vH3/6SJAG4dyj+zXECxnrQw8sA6V0oeESUmLpMSvcWkOHQiMdYVhpxwoPkjNJ0ucMPczBVAAZx498+AOcz9HaRLMNrVIZ13U8EqK6aFyYTSj1ZWY486QMqPPRgP14XT0Bbj67vwCfYLOEHHj99tt42IK2BnotPIyhBEn0wO5YHoreSaRegCB7wlyEBMrfMnqtLiz7XBV3nXjUc39y5aE5aVGi+c2LEJ7eqKrZ7WyIG5DUYzVrb1zNfV9Xp3ldRaR2emSUXCtT81h2mkUijmMTIu/EiYT4Q6nj0XFc8LS+vQ4i8QtJYjIcwHso9TLbR3Kpcqyl9006QF7pIsfw9wAqbTjNPIETbzs2/q32r8n6o4Onkfb4J7AtVnb0gxZqR6MYaqnxinpW273h2i89As8u5TOBvkXX7G1l+PQCaBneVRGjQ4Wg3Zkbu+cS9BDd+Vcqs18f7h9eo+VS4vB+8jg/eVsu2TvRMq/8FgnIugZJbzdQQgOjj7JpQoKgDY4yFVSy43/qawt0nVFEvK8Wtj7MSelI52eRlScysM+9T88oGsVhCC2YDrdLuhWVt02RO87OOVN0DD4Xfs1xzAHiwIQSAdhUavPemyuDYhvzE5XcnzzOttVrJo2jnYWMeqRLaPZRTH0QiEECnDoYAJD2FZpdGq3DDSOVq8INATja1GEeZKY4cHa3ndXb9GD1CxKRGl20CWruwuroWRCLDu6GhhHdYh0WFKg1knS3REBYQsjQHuPOY4hZxqcPd/cJK7oTVWCO+bCZ7sBCkjeQ8qd1XaQLwTuclSI7h0MoXQZFR+pdNov+OH4dnS9cmDVhorB1HFxTQgA5qwX2HnrBCyK88RDdzXcw3LsWAJdsrOMpKM/WSW7cTNmgxip5I1hgriF3Lttkxt8RQnik9cyuCeZUJESgNzqImoHcttPdG0Ec4OunQ16JtKi6tUlXUeP7wwWZ9Z2SeGb5UOz3t21hqydwqxnvNUmHyyTWjjMmgkmN1M0g23YKwP80Qxuiq5RHFauckxUTvUxMtaB0Ugs4Lfua2/koHZu2FQVZ3vf5uPe6PIuIz+93/rv/eWWpXUCDu/eVFz3ybQEqxLYtD9nB7Onz58dHBwcec9WW7nDBw4ivmG/fdIEPtob3sSg6yosbu43kHaPnXnJb1DcVQkWgBsbbordLhWIVOSi2jnL2VzFUG27ZaJTs/NvECRpqCo+mhv5/yZ3oaubdYH1NbdlgnYZ/qS7mNpMLxYwfPPNw5w9eTJxoD0Pnq3ybk+ivfmh/H4A1q2oK3foF+zrFK6zWsGyK+GgY1hcY74hMLLHh4Xc5DcBMLiMBaUlmKHwyrd7Dot8hT9aiyhD5DddWuFAsE2iRHoQiPAr6WK0VCGv80uNAxIYS0aQwz2xmbG/swPv4IcxqRx0603zhZn1WA2mQpXqUKmOVaczpUsp8cDu23dlXNIAY9+fWhXweYPK78s7WwMW/3BCVFJ+GzTLe4EafgOITQQzZ4cHPabazT1TxzXGp9P5c3SUtavh1wXbuRbk2nB2scAahSUK5oSuGQvf9srmzKr2+yZgGsE11e9lTY/CR/EcX++NtiRms9lo4sPYhU5jIp7wlN3r5sT+wFzsL4+8w912XWfbp/RDbXrJxDCxHufTisOekEm0wl8599VdAWkuvs28gNKVINgesXsa2u4OYL7CLzYa9l/4VzsR111SKqu2B6/7LCloinE8dK2uVJEiPXvH5ZrM/zl9cSOhaoGoNEEQ5WEQzKyVHo+igOslrrNYGKkH4pBvPupJc9cssg5qx3rT7ccrZzlYaOk3d9AeD6TT1DVwJi5FoTtYceCrr+ZMAqfbG2Z7VInNUN9953unhXN5MMcbDc+Wk6DAmXwrn8BiLcD8LFqL1dLT7PgQqHh+YG0PBzYiKXznVBf5kzuWY2Uqq3JqNDw/0CBZnVFtAZudeYauBvyNrKZ5ozPLRZvwWptTiF/HfJ3lsorDI2ayW2ypHJ+cojfax/4DDysGhkNdP6OaPZHPyKPp3HiaEFMEWJhOHdFB61tuvPQadiD2Xzh4uLabfMiu7rD2hgX+szeMp8CwFbgoda/CkFQKVCfJXk7To3ubaL7J/fmgojlGC3DO0ytN9cHm5fQZNPWJMfarFkarIAt0yLgvZrWnqI00TRbkpwtVfLY1DEbB+heL68MfQJMl9a/tsdtEDdsaN1gzjZdy+oXJ6ce2eB7j1SFW1FfgXDcCW3VVvEKZMqteNYpGUQkefpP2TKPXNe/OnuuKqs1Uy8ZYopDd/eH3aTOgClvjwmF2RdcNtSywk/PPpkSGXXbYSA1RyRsYVbGuxod2CS/8KbT0g4hNZ3q0oz7eJe+kShTedKtuo93zkXnGVZGDbqPJEQ+tiaZRF0MM5YHH+1zmdilZizHyYAJo+w7oA3uBBTqeUTcPL1yhhezcuCUM5qZ7g7I5nUHttI+tvSSvO4iWmj+AZFC0bL70ql0PUdERepuSnoaRhjQadA0hhjGCPVJH8A+LnE3JTvWpOrnWGJTOfTBdbHQ6t1J7EL/ze/WzfjOlwwNE1l5Bp7umlMAw28D1wyvUKLzl3onyR9u19hVNGlC3ijts6xVjbbZZdxAGgq5vOo90X+iy+2zXsb/WaIzrbQ6evC7KeuuLrQPWd6ZjSRVmVO5BAuZi7EEeVl01opuH1GmWqziDaLa/pBP+6gUH7Od+TorXtIkHvSSNBNO0o6aaUF0G/ANWNAkrGiK8uo2RT3PtYRBcQZKWNjeXbFT9tHHJxkkdLNQ9UWQ+Or+oXam+DBY04tzObIYGk7vNk3aFHn/kTQEb4khzbsTKdi477zyZx/X+doE3JWZdhdV3oihextqlbtRZbU8rkbKr4S1dJpeyLpyNmeCJRvHOi2tfm5FaRP3fstKvY8KDyenjwe3tAUcNpSRvQYCxcxA45r8CYV669x8pVWlATTYAAA=="
  },
  "guitar_articulation.py": {
    "sha256": "ec679e50b1a6e042beb973377d3ee79956e3ceaf8648cabdf84d0d16f8741c3d",
    "gzip_base64": "H4sIAAAAAAAC/51YSW/bRhS++1dMfSHZyLRjIwmqgocsThGgaAM07aGCQIzIR2kiksPMDCW7qf9735sZLtrsJjpYmu2t39t8fn7+aStZxU22gpzdXF1oyGSds2UrDFfMKMFLPWVSiaWoecm0UW1ViXrJNqB0q1kjsjW+rKUBHZ+fn5+JqpHKMK6WDVcauvVnLeuzQsmKNdysSrFg/uAjLrtL/4imECWcdWsnRdoouQDGtV+fnZ3lULBMVsgBQiWlmTAtW5XBhMnWNC2uf8QtvGuSV6/i5zdXL69fTBjcQdYaSN6jThBNzxh+jr1mCZuRWGETxQq0LDcQRqyQijVM1Owox2huyTkJY40WKiHmxvBsZe9H9tgZQMkN1LzOoLOBXvHrFy8nbKuEgdSaim47ynG1zoUKSdfa6OSTaoFUEdqkcu11caqAJZawrwGqblodTFnQ4C5XKE4wYYETOXXs8NT9CN12NLFUhk9gDZg6RBAxu0Yyeau4EbIeHd1cHbyuZA5lmvGy1KmCLy1oAzlevT64SeAhWf/gFYww2AheSzQ4eYQTJqEGx5hpMAZ10j8zWZf3LOON3c5WvF4iDtlbhw1UEC0CasAvX3JRa8PMCtBenyFDmdhPPUvV1mwB6GmEm0HwL1pig+gWGpiRiLnawJ1hJdRLs4rZb5IVbVleoMeWSG+QMA4erJaDR0OPrUsWOOQK3IzpBD3jXOfcaNT9tLeQC0Dy6UO/R0CsyVSExaBTjfzrgjGIpjsWdjRm9GSOlDxEvazH4welpOtdDDnH77nt0U8HkQSB0ced/56wDWrPa5MQj2iH7P+0Vy4yEzqjTWwiKAEdmTpNkxJjI3S/o2igzydsgfp7cwyGm0/6PW/Aef9GFIzPAovBYM5+SNhiWKEbhrNZHxXu3s3VrhMUJwj9xcsWbpWSKgze9loxCg+BmcbCUpNrO51G4eAYRXuiIRBF3Yd0J+L+Loq6G+t0z9lvtpcV5t8gt2ND1IcCYYn5SMxH4mIiwqhNc1EUgJksA0K1pvDMw7XF9JoAjSq5m8E8XsO9xsT7L2m0v+m073eXYMJ15LXf290x2REx8NEs8CmEokjkTxvhd0o75C1fKP3zS0eeiRxr6j1zXEZGcCaP2yanyLMe9XZPxlDa88jkiNTJ4daOnphTu7jb02VHBFcokqApeV3v+MvdNa2q/ZM94PkSYxEXdMkkDwgMi5OnT5n1jTSr3d6DVS2m60LUQq8OwN+VAbIZxkghln0EHDt6ivsvjvGozriX+tCRDSaThrLJnhQYGbzS5LLF8YOewlOB0PAB/81iBHs8GNCOJ0dA3qPZwgCJjZlRLsAnY+CwC+wZRhGgVZbyNhcyeHjKZn/WcNe4MjpmsgCzBcBwzkyLWcG7tDfIqYjo4DggZuJJpNaAmBHVThSMfp+qT11qSrH/qEbRNlraELOpf+fSYv/SCQ4IUYF9TvoF5RTmPgnaWsFGwHYnoL6jFeiqPTXFrh5Tt0EOnYWBb46xMousLV1DVgDkC56tY+ykkZZrDifs+GVMIG1p/FVqK6O9CNkKjEbfk8d/i+Y9fg+yD0IF28BVYaSoqej3bz58TN/dvv/19afbdxE18FxlK7HZy0i2NruD2BrpUfscbEUHxL61QRqwSJbZ2ICcBePho2fu2mnIL22u7rf7u9btcdXc0G4l7na25ke5YrySV6cn26tBrGeDXPGWbzwP+/M4bbJE95ys0ZM6ze2EJ6w9L3sCE7fxjAWXAf7ttsdxvV864C6DxrA3XMOt/UkJFjEBlEimTySEgiOeKBvY20kRfDX3DYR2FcVpStKk6cOUfbVbD+MMQznLTV8Eg3LUXX9jSOJcWrCOF0uwsqVphfNEmvrCZideRVXBT7/xa7VsK5zbPtqTMAedKWFVT9I0l1maRqOXMc9zjFH3JAwuLqg3RwFI14QmUux8oeAYtXYVpimFGdIYxtTYz4mz5/NHKbsWY5e2b0NzO2M++hqt9b1P7SzRPS5Kyc2gVD+rP0rBtzWuNJAlMUvjxIbdfwve73idgtgTsF9EAiuoI4wDsQnJu3HeVo0Ou/8l0J3YDUT2ZzcV2YWk/yu4gcgdunG4m27snl9gzhV1jvIm11iZ/wPdRaJAZhEAAA=="
  }
}
# END EMBEDDED CONTEXT HELPERS
TOOLS = Path(tempfile.mkdtemp(prefix="guitar_articulation_tools_"))
for name, payload in EMBEDDED_HELPERS.items():
    data = gzip.decompress(base64.b64decode(payload["gzip_base64"], validate=True))
    assert hashlib.sha256(data).hexdigest() == payload["sha256"], f"Helper checksum mismatch: {name}"
    (TOOLS / name).write_bytes(data)
spec = importlib.util.spec_from_file_location("electric_full_song", TOOLS / "electric_full_song.py")
single = importlib.util.module_from_spec(spec); spec.loader.exec_module(single)
sys.modules["electric_full_song"] = single
ROOT = single.discover_root(Path.cwd(), globals().get("ROOT"))
if ROOT is None and Path('/content').is_dir():
    ROOT = single.discover_root(Path('/content'))
if ROOT is None:
    raise RuntimeError("Source bridge belum ditemukan. Gunakan VM aktif dari notebook 12 atau jalankan sel setup notebook 12 sampai bobot siap")
if not DRY_RUN:
    single.model_python(ROOT)
spec = importlib.util.spec_from_file_location("guitar_probe_embedded", TOOLS / "guitar_probe.py")
guitar = importlib.util.module_from_spec(spec); spec.loader.exec_module(guitar)
sys.modules["guitar_probe"] = guitar
spec = importlib.util.spec_from_file_location("guitar_articulation_embedded", TOOLS / "guitar_articulation.py")
articulation = importlib.util.module_from_spec(spec); spec.loader.exec_module(articulation)
SOURCE_PATH = globals().get("SOURCE_PATH")
if SOURCE_PATH is None or not Path(SOURCE_PATH).is_file():
    uploaded = files.upload()
    audios = [n for n in uploaded if Path(n).suffix.lower() in {".mp3", ".wav", ".flac", ".m4a", ".aac", ".ogg"}]
    assert len(audios) == 1, "Upload satu piano asli"
    SOURCE_PATH = TOOLS / ("piano" + Path(audios[0]).suffix.lower())
    SOURCE_PATH.write_bytes(uploaded[audios[0]])
print("Runtime dipakai kembali:", ROOT)
print("Piano:", SOURCE_PATH)


## Bandingkan A/B pada bagian yang sama

`START` memakai awal input chunk 1, bukan batas core di 83.13 detik. Durasi tetap 30 detik. Dengarkan noise, serangan/pelepasan nada, akor dan timing. Jika original 30 detik lebih bersih daripada chunk panjang pada bagian ini, itu mendukung dugaan pengaruh panjang konteks, tanpa membuktikan penyebab tunggal. Perbandingan original versus picked menguji perubahan artikulasi pada panjang konteks yang sama.


In [ ]:
from IPython.display import Audio, display
START = 77.130625
ARTICULATION_OUT = ROOT / "runs" / ("guitar_articulation30_" + uuid.uuid4().hex[:10])
report = articulation.compare(ROOT, SOURCE_PATH, ARTICULATION_OUT, start=START, execute=not DRY_RUN)
print("Status:", report["status"], "|", ARTICULATION_OUT)
if not DRY_RUN:
    print("Perbedaan parameter:", report["actual_parameter_differences"])
    for variant in ("original", "picked"):
        for filename in ("guitar_preview.mp3", "mix_preview.mp3"):
            print(variant, filename)
            display(Audio(filename=str(ARTICULATION_OUT / variant / filename), normalize=False))


## Simpan dan kirim feedback

Simpan `guitar_articulation_result.zip` untuk WAV lossless. Kirim `guitar_articulation_feedback.zip`, lalu jelaskan apakah original atau picked lebih bersih dan lebih alami. Belum ada perubahan baseline atau generasi full-song otomatis dari eksperimen ini.


In [ ]:
for name in ("guitar_articulation_result.zip", "guitar_articulation_feedback.zip"):
    path = ARTICULATION_OUT / name
    if path.is_file():
        files.download(str(path))
if DRY_RUN:
    print("Plan saja; belum ada audio model atau ZIP hasil")
